Importación de paquetes necesarios

In [18]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from joblib import Parallel, delayed
from rdkit.Chem import MACCSkeys
from sklearn.feature_selection import VarianceThreshold

Importación del dataset curado con los compuestos para el entrenamiento de los modelos

In [19]:
df_limpio = pd.read_csv('../data/processed/chembl_curado.csv')
df_limpio.head()

,Smiles,Standard Relation,pChEMBL Value
0,C#CN1C[C@@H]2CN(C(=O)C=Cc3cnc4c(c3)CCC(=O)N4)C...,'=',5.10
1,C=C(C)COC1CN(C(=O)C=Cc2cnc3c(c2)CCC(=O)N3)C1,'=',6.77
2,C=CC(C)(C)C12CC(=O)C(=O)N3C(=Cc4c[nH]cn4)C(=O)...,'=',4.40
3,CC(=O)Nc1ccc(C=CC(=O)N(C)Cc2cc3ccccc3n2C)cn1,'=',6.52
4,CC(=O)Nc1ccc(C=CC(=O)N2CC(OCc3ccncc3)C2)cn1,'=',6.58


For this exercise we will use a subset of 78 descriptors used in this NCATS article https://pubs.acs.org/doi/10.1021/acs.jcim.0c00884

In [20]:
lista_descriptores = [
    "MolLogP", "MolWt", "TPSA", "LabuteASA", "HeavyAtomMolWt", "ExactMolWt",
    "NumHAcceptors", "NumHDonors", "NumRotatableBonds", "NumHeteroatoms",
    "HeavyAtomCount", "NHOHCount", "NOCount", "NumAliphaticCarbocycles",
    "NumAliphaticHeterocycles", "NumAliphaticRings", "NumAromaticCarbocycles",
    "NumAromaticHeterocycles", "NumAromaticRings", "NumSaturatedCarbocycles",
    "NumSaturatedHeterocycles", "NumSaturatedRings", "RingCount",
    "FractionCSP3", "Chi0v", "Chi1v", "Chi2v", "Chi3v", "Chi4v", "Chi1n",
    "Chi2n", "Chi3n", "Chi4n", "HallKierAlpha", "Kappa1", "Kappa2", "Kappa3",
    "SlogP_VSA1", "SlogP_VSA2", "SlogP_VSA3", "SlogP_VSA4", "SlogP_VSA5",
    "SlogP_VSA6", "SlogP_VSA7", "SlogP_VSA8", "SlogP_VSA9", "SlogP_VSA10",
    "SlogP_VSA11", "SlogP_VSA12", "SMR_VSA1", "SMR_VSA2", "SMR_VSA3", "SMR_VSA4",
    "SMR_VSA5", "SMR_VSA6", "SMR_VSA7", "SMR_VSA8", "SMR_VSA9", "SMR_VSA10",
    "PEOE_VSA1", "PEOE_VSA2", "PEOE_VSA3", "PEOE_VSA4", "PEOE_VSA5", "PEOE_VSA6",
    "PEOE_VSA7", "PEOE_VSA8", "PEOE_VSA9", "PEOE_VSA10", "PEOE_VSA11", "PEOE_VSA12",
    "PEOE_VSA13", "PEOE_VSA14"
]

Next, we need to set up a calculator to calculate the descriptors. The ML.Descriptors module has a MolecularDescriptorsCalculator to achieve this.

In [21]:
calc = MoleculeDescriptors.MolecularDescriptorCalculator(lista_descriptores)

Once the calc variable is set up, we can calculate all descriptors for a given SMILES. Let's iterate over each SMILES string in our dataset and return the descriptors. We will use paralellisation to speed this up.

In [22]:
def descriptors_from_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    desc = calc.CalcDescriptors(mol)
    return list(desc)

all_smiles = df_limpio['Smiles'].tolist()
all_descriptors = Parallel(n_jobs=-1, prefer='threads')(delayed(descriptors_from_smiles)(smi) for smi in all_smiles)

Now we want the descriptors in the format of a DataFrame for Machine Learning, where each
row is a data point (compound) and each column is a descriptor. To do this, we just call
pd.DataFrame on the list of calculated descriptors, and each item in the list will be transformed
into a row of the DataFrame.

In [23]:
desc_df = pd.DataFrame(all_descriptors)
desc_df.head()

,0,1,2,3,4,5,6,7,8,9,...,63,64,65,66,67,68,69,70,71,72
0,0.9604,336.395,65.54,146.444943,316.235,336.158626,4,1,2,6,...,0.000000,6.423350,29.690112,62.752892,0.000000,5.817863,0.0,11.814359,0.000000,0.0
1,1.7830,327.384,71.53,140.750715,306.216,327.158292,4,1,5,6,...,0.000000,12.152040,36.613849,31.783198,12.710848,5.817863,0.0,11.814359,0.000000,0.0
2,1.8672,433.468,107.63,184.251475,410.284,433.175004,6,2,4,9,...,6.578936,38.122596,23.120829,12.617665,30.233422,5.697039,0.0,11.570040,11.814359,0.0
3,3.2035,362.433,67.23,157.671616,340.257,362.174276,3,1,5,6,...,0.000000,18.199101,41.291164,44.502574,6.544756,5.817863,0.0,11.814359,0.000000,0.0
4,1.8758,352.394,84.42,151.263123,332.234,352.153541,5,1,6,7,...,0.000000,0.000000,41.468391,44.679801,12.710848,5.817863,0.0,11.814359,0.000000,0.0


To understand feature importance later, we need to set the DataFrame column names to
correspond to each descriptor. Since we already have the descriptor names stored in a list, we
assign the .columns attribute of the DataFrame to the list.

In [24]:
desc_df.columns = lista_descriptores
desc_df.head()

,MolLogP,MolWt,TPSA,LabuteASA,HeavyAtomMolWt,ExactMolWt,NumHAcceptors,NumHDonors,NumRotatableBonds,NumHeteroatoms,...,PEOE_VSA5,PEOE_VSA6,PEOE_VSA7,PEOE_VSA8,PEOE_VSA9,PEOE_VSA10,PEOE_VSA11,PEOE_VSA12,PEOE_VSA13,PEOE_VSA14
0,0.9604,336.395,65.54,146.444943,316.235,336.158626,4,1,2,6,...,0.000000,6.423350,29.690112,62.752892,0.000000,5.817863,0.0,11.814359,0.000000,0.0
1,1.7830,327.384,71.53,140.750715,306.216,327.158292,4,1,5,6,...,0.000000,12.152040,36.613849,31.783198,12.710848,5.817863,0.0,11.814359,0.000000,0.0
2,1.8672,433.468,107.63,184.251475,410.284,433.175004,6,2,4,9,...,6.578936,38.122596,23.120829,12.617665,30.233422,5.697039,0.0,11.570040,11.814359,0.0
3,3.2035,362.433,67.23,157.671616,340.257,362.174276,3,1,5,6,...,0.000000,18.199101,41.291164,44.502574,6.544756,5.817863,0.0,11.814359,0.000000,0.0
4,1.8758,352.394,84.42,151.263123,332.234,352.153541,5,1,6,7,...,0.000000,0.000000,41.468391,44.679801,12.710848,5.817863,0.0,11.814359,0.000000,0.0


When calculating descriptors its possible that some calculations will fail, resulting in null values
for some compound-descriptor combinations. To remove columns where there are null values
present in any row, we can use dropna again. This time, we pass axis=1 to indicate that we want
to drop columns, not rows.

In [25]:
df_fisicoquimicos = desc_df.dropna(axis=1, how='any')
df_fisicoquimicos.shape

(344, 73)

Unimos la variable objetivo al nuevo DataFrame 

In [26]:
df_fisicoquimicos['pChEMBL_Value'] = df_limpio['pChEMBL Value']
df_fisicoquimicos.head()

,MolLogP,MolWt,TPSA,LabuteASA,HeavyAtomMolWt,ExactMolWt,NumHAcceptors,NumHDonors,NumRotatableBonds,NumHeteroatoms,...,PEOE_VSA6,PEOE_VSA7,PEOE_VSA8,PEOE_VSA9,PEOE_VSA10,PEOE_VSA11,PEOE_VSA12,PEOE_VSA13,PEOE_VSA14,pChEMBL_Value
0,0.9604,336.395,65.54,146.444943,316.235,336.158626,4,1,2,6,...,6.423350,29.690112,62.752892,0.000000,5.817863,0.0,11.814359,0.000000,0.0,5.10
1,1.7830,327.384,71.53,140.750715,306.216,327.158292,4,1,5,6,...,12.152040,36.613849,31.783198,12.710848,5.817863,0.0,11.814359,0.000000,0.0,6.77
2,1.8672,433.468,107.63,184.251475,410.284,433.175004,6,2,4,9,...,38.122596,23.120829,12.617665,30.233422,5.697039,0.0,11.570040,11.814359,0.0,4.40
3,3.2035,362.433,67.23,157.671616,340.257,362.174276,3,1,5,6,...,18.199101,41.291164,44.502574,6.544756,5.817863,0.0,11.814359,0.000000,0.0,6.52
4,1.8758,352.394,84.42,151.263123,332.234,352.153541,5,1,6,7,...,0.000000,41.468391,44.679801,12.710848,5.817863,0.0,11.814359,0.000000,0.0,6.58


Ahora vamos a crear otro df esta vez utilizando descriptores estructurales MACCS Keys.

In [27]:
def calcular_maccs(smi):
    mol = Chem.MolFromSmiles(smi)
    maccs = MACCSkeys.GenMACCSKeys(mol)
    return list(maccs)

maccs_listas = df_limpio['Smiles'].apply(calcular_maccs).tolist()

In [28]:
nombres_columnas_maccs = [f"MACCS_{i}" for i in range(167)]
df_maccs = pd.DataFrame(maccs_listas, columns=nombres_columnas_maccs)
df_maccs.head()

,MACCS_0,MACCS_1,MACCS_2,MACCS_3,MACCS_4,MACCS_5,MACCS_6,MACCS_7,MACCS_8,MACCS_9,...,MACCS_157,MACCS_158,MACCS_159,MACCS_160,MACCS_161,MACCS_162,MACCS_163,MACCS_164,MACCS_165,MACCS_166
0,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,1,1,1,0
1,0,0,0,0,0,0,0,0,1,0,...,1,1,1,1,1,1,1,1,1,0
2,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
3,0,0,0,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,1,1,0
4,0,0,0,0,0,0,0,0,1,0,...,1,1,1,1,1,1,1,1,1,0


In [29]:
df_maccs['pChEMBL_Value'] = df_limpio['pChEMBL Value']
df_maccs.head()

,MACCS_0,MACCS_1,MACCS_2,MACCS_3,MACCS_4,MACCS_5,MACCS_6,MACCS_7,MACCS_8,MACCS_9,...,MACCS_158,MACCS_159,MACCS_160,MACCS_161,MACCS_162,MACCS_163,MACCS_164,MACCS_165,MACCS_166,pChEMBL_Value
0,0,0,0,0,0,0,0,0,0,0,...,1,1,0,1,1,1,1,1,0,5.10
1,0,0,0,0,0,0,0,0,1,0,...,1,1,1,1,1,1,1,1,0,6.77
2,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,0,4.40
3,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,0,6.52
4,0,0,0,0,0,0,0,0,1,0,...,1,1,1,1,1,1,1,1,0,6.58


Una vez tenemos ambas tablas, es matemáticamente obligatorio aplicar dos filtros para reducir su dimensionalidad, mejorar la precisión del modelo y evitar el sobreajuste (overfitting):

Filtro de Varianza (< 0.1): Elimina aquellos descriptores que son casi idénticos para todos tus compuestos (por ejemplo, fragmentos de MACCS que valen "0" en casi todas las filas). Si no varían, no aportan información útil para discriminar qué hace que una molécula sea activa o inactiva.
Filtro de Correlación de Pearson (> 0.8): Elimina variables que miden exactamente lo mismo (colinealidad). Si dos descriptores fisicoquímicos tienen una correlación superior al 80%, el código eliminará uno de ellos para no saturar al algoritmo con información redundante.

1. Definimos una función general para filtrar cualquier DataFrame:

In [30]:
def filtrar_features(df, nombre_dataset, var_threshold=0.1, corr_threshold=0.8):
    print(f"--- FILTRANDO {nombre_dataset} ---")
    # Separar la columnas que NO son descriptores para que no se filtren
    cols_meta = ['pChEMBL_Value']
    df_descriptores = df.drop(columns=cols_meta)
    print(f"Descriptores iniciales: {df.shape[1] - 1}") 
    #FILTRO DE VARIANZA
    selector = VarianceThreshold(threshold=var_threshold)
    selector.fit(df_descriptores)
    cols_filtradas_var = df_descriptores.columns[selector.get_support()]
    df_var = df_descriptores[cols_filtradas_var]
    print(f"Descriptores tras Filtro de Varianza (< {var_threshold}): {df_var.shape[1]}")
    # B. FILTRO DE CORRELACIÓN DE PEARSON
    # Calculamos la matriz de correlación en valor absoluto
    matriz_corr = df_var.corr(method='pearson').abs()
     # Seleccionamos solo el triángulo superior de la matriz para no evaluar pares duplicados
    upper = matriz_corr.where(np.triu(np.ones(matriz_corr.shape), k=1).astype(bool))
    # Encontramos las características con una correlación superior al umbral
    to_drop = [column for column in upper.columns if any(upper[column] > corr_threshold)]
    df_final = df_var.drop(columns=to_drop)
    print(f"Descriptores tras Filtro de Correlación (> {corr_threshold}): {df_final.shape[1]}\n")
    # Volvemos a unir las características limpias con nuestra variable objetivo y SMILES
    df_filtrado_final = pd.concat([df_final, df[cols_meta]], axis=1)
    return df_filtrado_final

2. Aplicamos la función a nuestros dos DataFrames

In [31]:
df_fisicoquimicos_filtrado = filtrar_features(df_fisicoquimicos, "FISICOQUÍMICOS", var_threshold=0.1, corr_threshold=0.8)
df_maccs_filtrado = filtrar_features(df_maccs, "MACCS KEYS", var_threshold=0.1, corr_threshold=0.8)

--- FILTRANDO FISICOQUÍMICOS ---
Descriptores iniciales: 73
Descriptores tras Filtro de Varianza (< 0.1): 69
Descriptores tras Filtro de Correlación (> 0.8): 34

--- FILTRANDO MACCS KEYS ---
Descriptores iniciales: 167
Descriptores tras Filtro de Varianza (< 0.1): 75
Descriptores tras Filtro de Correlación (> 0.8): 49



3. Exportamos los datasets definitivos listos para el entrenamiento

In [ ]:
ruta_fisicoq = '../data/processed/fisicoquimicos_filtrados.csv'
ruta_maccs = '../data/processed/maccs_filtrados.csv'

df_fisicoquimicos_filtrado.to_csv(ruta_fisicoq, index=False)
df_maccs_filtrado.to_csv(ruta_maccs, index=False)
print("¡Archivos exportados exitosamente y listos para entrenar los modelos!")

¡Archivos exportados exitosamente y listos para entrenar los modelos!
